In [1]:
import pdfplumber
from pathlib import Path

CAMINHO_PDF = Path("../data/raw/anuario_cerveja_2026.pdf")

with pdfplumber.open(CAMINHO_PDF) as pdf:
    print("Total de páginas:", len(pdf.pages))

    for numero_tabela in range(16, 24):
        rotulo = f"Tabela {numero_tabela}:"
        paginas_encontradas = []

        for indice, pagina in enumerate(pdf.pages):
            texto = pagina.extract_text() or ""
            if rotulo in texto:
                paginas_encontradas.append(indice)

        print(rotulo, "índices", paginas_encontradas)

Total de páginas: 64
Tabela 16: índices [47]
Tabela 17: índices [49]
Tabela 18: índices [50]
Tabela 19: índices [51]
Tabela 20: índices [51]
Tabela 21: índices [54]
Tabela 22: índices [55]
Tabela 23: índices [55]


In [2]:
with pdfplumber.open(CAMINHO_PDF) as pdf:
    pagina = pdf.pages[51]

    print("===== TEXTO =====")
    print(pagina.extract_text())

    tabelas = pagina.extract_tables()
    print("===== TABELAS DETECTADAS:", len(tabelas), "=====")

    for numero, tabela in enumerate(tabelas, start=1):
        print("--- tabela", numero, "---")
        for linha in tabela:
            print(linha)

===== TEXTO =====
A região nordeste, embora ocupe a terceira posição em números de cervejarias registradas,
com 130 estabelecimentos, ocupa a segunda posição quanto ao volume de produção
declarado, com 3.802.222.465,12 litros, o que representa 24,2% do volume nacional.
A região Sul, por sua vez, embora ocupando a segunda colocação quanto ao número de
cervejarias registradas, com 759 estabelecimentos, ocupa a terceira posição quanto ao
volume de produção declarado, com 2.447.510.384,25, o que representa 15,6% do volume
nacional.
Tal contradição deve-se à produção média verificada em cada região, que se encontram
em extremos opostos, sendo a da região Nordeste a maior, com 29.247.865,12 litros por
cervejaria, enquanto aquela da região sul é a menor, com 3.224.651,36 litros de cerveja
por estabelecimento.
Tabela 19: Análise da concentração do volume produzido de cerveja no Brasil
Porcentagem de Cervejarias Porcentagem relativa ao
Volume produzido
declarantes volume total produzido
1% 6.66

In [3]:
import pandas as pd

def converter_numero_br(texto):
    limpo = texto.replace("%", "").replace(".", "").replace(",", ".").strip()
    return float(limpo)

print(converter_numero_br("6.666.193.835,38"))
print(converter_numero_br("42,49%"))
print(converter_numero_br("-3,90"))

6666193835.38
42.49
-3.9


In [6]:
pd.options.display.float_format = "{:,.2f}".format

In [7]:
with pdfplumber.open(CAMINHO_PDF) as pdf:
    tabelas = pdf.pages[51].extract_tables()

linhas = tabelas[0][1:]

df_19 = pd.DataFrame(linhas, columns=["pct_cervejarias", "volume_litros", "pct_volume"])

for coluna in df_19.columns:
    df_19[coluna] = df_19[coluna].map(converter_numero_br)

df_19["ano_referencia"] = 2025
df_19["fonte"] = "Anuário da Cerveja 2026, Tabela 19, p. 52"

PASTA_SAIDA = Path("../data/processed")
df_19.to_csv(PASTA_SAIDA / "tabela_19_concentracao_2025.csv", index=False)

df_19

,pct_cervejarias,volume_litros,pct_volume,ano_referencia,fonte
0,1.00,"6,666,193,835.38",42.49,2025,"Anuário da Cerveja 2026, Tabela 19, p. 52"
1,2.00,"11,238,768,122.78",71.64,2025,"Anuário da Cerveja 2026, Tabela 19, p. 52"
2,3.00,"13,973,317,068.01",89.07,2025,"Anuário da Cerveja 2026, Tabela 19, p. 52"
3,4.00,"15,150,315,200.57",96.57,2025,"Anuário da Cerveja 2026, Tabela 19, p. 52"
4,5.00,"15,471,743,939.70",98.62,2025,"Anuário da Cerveja 2026, Tabela 19, p. 52"


In [8]:
linhas = tabelas[1][1:]

df_20 = pd.DataFrame(linhas, columns=["categoria", "volume_litros", "pct_volume", "variacao_pct"])

df_20["categoria"] = df_20["categoria"].str.replace("\n", " ")

for coluna in ["volume_litros", "pct_volume", "variacao_pct"]:
    df_20[coluna] = df_20[coluna].map(converter_numero_br)

df_20["ano_referencia"] = 2025
df_20["fonte"] = "Anuário da Cerveja 2026, Tabela 20, p. 52"

df_20.to_csv(PASTA_SAIDA / "tabela_20_puro_malte_2025.csv", index=False)

print("Soma dos volumes:", df_20["volume_litros"].sum())
df_20

Soma dos volumes: 15688083191.69


,categoria,volume_litros,pct_volume,variacao_pct,ano_referencia,fonte
0,Cerveja Puro Malte ou 100% malte,"4,580,310,971.90",29.20,21.01,2025,"Anuário da Cerveja 2026, Tabela 20, p. 52"
1,Demais cervejas,"11,107,772,219.79",70.80,-3.90,2025,"Anuário da Cerveja 2026, Tabela 20, p. 52"


In [10]:
with pdfplumber.open(CAMINHO_PDF) as pdf:
    for indice in [49, 50]:
        tabelas = pdf.pages[indice].extract_tables()
        print("===== PÁGINA ÍNDICE", indice, "- tabelas detectadas:", len(tabelas), "=====")

        for posicao, tabela in enumerate(tabelas):
            print("--- posição", posicao, "---")
            for linha in tabela:
                print(linha)

===== PÁGINA ÍNDICE 49 - tabelas detectadas: 2 =====
--- posição 0 ---
['']
['']
--- posição 1 ---
['Região', 'Volume (L)', 'Proporção relativa ao\nvolume nacional (%)', 'Variação 2024/2025 (%)']
['Centro-Oeste', '1.054.248.799,25', '6,7', '-5,59']
['Nordeste', '3.802.222.465,12', '24,2', '-8,07']
['Norte', '215.667.721,98', '1,4', '-16,92']
['Sudeste', '8.168.433.821,09', '52,1', '-9,36']
['Sul', '2.447.510.384,25', '15,6', '-8,90']
===== PÁGINA ÍNDICE 50 - tabelas detectadas: 1 =====
--- posição 0 ---
['Região', 'Volume (L)', 'Nº de cervejarias', 'Produção Média (L/\ncervejaria)']
['Nordeste', '3.802.222.465,12', '130', '29.247.865,12']
['Centro-Oeste', '1.054.248.799,25', '98', '10.757.640,81']
['Norte', '215.667.721,98', '44', '4.901.539,14']
['Sul', '2.447.510.384,25', '759', '3.224.651,36']
['Sudeste', '8.168.433.821,09', '923', '8.849.874,13']


In [11]:
with pdfplumber.open(CAMINHO_PDF) as pdf:
    tabela_17 = pdf.pages[49].extract_tables()[1]
    tabela_18 = pdf.pages[50].extract_tables()[0]

df_17 = pd.DataFrame(tabela_17[1:], columns=["regiao", "volume_litros", "pct_volume", "variacao_pct"])
for coluna in ["volume_litros", "pct_volume", "variacao_pct"]:
    df_17[coluna] = df_17[coluna].map(converter_numero_br)
df_17["ano_referencia"] = 2025
df_17["fonte"] = "Anuário da Cerveja 2026, Tabela 17, p. 50"
df_17.to_csv(PASTA_SAIDA / "tabela_17_producao_regiao_2025.csv", index=False)

df_18 = pd.DataFrame(tabela_18[1:], columns=["regiao", "volume_litros", "n_cervejarias", "producao_media_litros"])
for coluna in ["volume_litros", "n_cervejarias", "producao_media_litros"]:
    df_18[coluna] = df_18[coluna].map(converter_numero_br)
df_18["n_cervejarias"] = df_18["n_cervejarias"].astype(int)
df_18["ano_referencia"] = 2025
df_18["fonte"] = "Anuário da Cerveja 2026, Tabela 18, p. 51"
df_18.to_csv(PASTA_SAIDA / "tabela_18_producao_media_regiao_2025.csv", index=False)

print("Soma das regiões (T17):", df_17["volume_litros"].sum())
print("Soma das regiões (T18):", df_18["volume_litros"].sum())
print("Total de cervejarias:", df_18["n_cervejarias"].sum())

display(df_17)
display(df_18)

Soma das regiões (T17): 15688083191.689999
Soma das regiões (T18): 15688083191.689999
Total de cervejarias: 1954


,regiao,volume_litros,pct_volume,variacao_pct,ano_referencia,fonte
0,Centro-Oeste,"1,054,248,799.25",6.70,-5.59,2025,"Anuário da Cerveja 2026, Tabela 17, p. 50"
1,Nordeste,"3,802,222,465.12",24.20,-8.07,2025,"Anuário da Cerveja 2026, Tabela 17, p. 50"
2,Norte,"215,667,721.98",1.40,-16.92,2025,"Anuário da Cerveja 2026, Tabela 17, p. 50"
3,Sudeste,"8,168,433,821.09",52.10,-9.36,2025,"Anuário da Cerveja 2026, Tabela 17, p. 50"
4,Sul,"2,447,510,384.25",15.60,-8.90,2025,"Anuário da Cerveja 2026, Tabela 17, p. 50"


,regiao,volume_litros,n_cervejarias,producao_media_litros,ano_referencia,fonte
0,Nordeste,"3,802,222,465.12",130,"29,247,865.12",2025,"Anuário da Cerveja 2026, Tabela 18, p. 51"
1,Centro-Oeste,"1,054,248,799.25",98,"10,757,640.81",2025,"Anuário da Cerveja 2026, Tabela 18, p. 51"
2,Norte,"215,667,721.98",44,"4,901,539.14",2025,"Anuário da Cerveja 2026, Tabela 18, p. 51"
3,Sul,"2,447,510,384.25",759,"3,224,651.36",2025,"Anuário da Cerveja 2026, Tabela 18, p. 51"
4,Sudeste,"8,168,433,821.09",923,"8,849,874.13",2025,"Anuário da Cerveja 2026, Tabela 18, p. 51"


In [12]:
with pdfplumber.open(CAMINHO_PDF) as pdf:
    for indice in [47, 54, 55]:
        tabelas = pdf.pages[indice].extract_tables()
        print("===== PÁGINA ÍNDICE", indice, "- tabelas detectadas:", len(tabelas), "=====")

        for posicao, tabela in enumerate(tabelas):
            print("--- posição", posicao, "---")
            for linha in tabela:
                print(linha)

===== PÁGINA ÍNDICE 47 - tabelas detectadas: 1 =====
--- posição 0 ---
['UF', 'Estoque de empregos', 'Porcentagem Relativa', 'Variação Relativa']
['RO', '45', '0,11', '0,00%']
['AC', '11', '0,03', '0,00%']
['AM', '531', '1,27', '-2,57%']
['RR', '13', '0,03', '0,00%']
['PA', '883', '2,10', '-4,13%']
['AP', '5', '0,01', 'de 0 para 5']
['TO', '14', '0,03', '-30%']
['MA', '694', '1,65', '-3,74%']
['PI', '448', '1,07', '-0,67%']
['CE', '648', '1,54', '-14,29%']
['RN', '45', '0,11', '-21,05%']
['PB', '59', '0,14', '-3,51%']
['PE', '2.288', '5,45', '-3,74%']
['AL', '40', '0,10', '8,11%']
['SE', '417', '0,99', '-5,44%']
['BA', '2.234', '5,32', '-0,18%']
['MG', '4.720', '11,24', '3,74%']
['ES', '240', '0,57', '18,91%']
['RJ', '5.216', '12,43', '-3,19%']
['SP', '14.182', '33,79', '-2,82%']
['PR', '2.680', '6,38', '-5,50%']
['SC', '1.552', '3,70', '-8,81%']
['RS', '2.479', '5,91', '-1,70%']
['MS', '35', '0,08', '-7,89%']
['MT', '906', '2,16', '-3,10%']
['GO', '1.535', '3,66', '-1,85%']
['DF', '56